# LocalFold, in Colab

Fold on the GPU Colab lends you, and look at the answer here. Two cells to set
up, then one cell you run as often as you like.

**Runtime → Change runtime type → T4 GPU** before you start.

---

🔴 **WHY THIS IS NOT THE WEBSITE IN A FRAME.** JavaScript in a Colab output
cell runs in YOUR BROWSER, not on the runtime - so a LocalFold embedded in a
cell would fold on your laptop exactly as localfold.org does, with the T4
idle. What runs on the runtime here is a headless Chrome: the same
`web/app.js`, on the card Colab lent you. Only the PICTURE is drawn locally.

The fold is LocalFold's own code either way. There is no second
implementation and no Python port of the model.

In [ ]:
#@title (written by cell 1)
%%writefile /content/_setup.sh
set -e
# 🔴 THE USERSPACE HALF OF THE DRIVER, MATCHED TO THE KERNEL HALF. A runtime
# ships the kernel module and `nvidia-smi`, not the Vulkan ICD a browser needs
# - and a Chrome that asks for Vulkan and does not find it is handed
# SwiftShader, the CPU renderer, which folds and means nothing. Measured on
# the first real runtime: vendor 'google', architecture 'swiftshader'.
DRIVER=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | cut -d. -f1)
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y libvulkan1 vulkan-tools dbus > /dev/null 2>&1
[ -n "$DRIVER" ] && apt-get -qq install -y "libnvidia-gl-${DRIVER}" > /dev/null 2>&1 || true
if ! command -v google-chrome > /dev/null; then
  wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null 2>&1
fi
if [ -d /content/localfold ]; then git -C /content/localfold pull -q --ff-only || true
else git clone -q --depth 1 https://github.com/sokrypton/localfold /content/localfold; fi
pip -q install py2Dmol
google-chrome --version
vulkaninfo --summary 2>/dev/null | grep -i deviceName | head -2 || echo "no Vulkan device: the fold would fall back to the CPU"

In [ ]:
#@title 1 · Set up (about two minutes) { display-mode: "form" }
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "NO GPU - Runtime > Change runtime type > T4"
!bash /content/_setup.sh 2>&1 | tail -6

In [ ]:
#@title 2 · Start the fold service on this runtime { display-mode: "form" }
import json, queue, subprocess, sys, threading, time

PORT, REPO = 8710, '/content/localfold'

def _drain(stream, sink):
    for line in iter(stream.readline, ''):
        sink.put(line.rstrip())

backend = subprocess.Popen(
    [sys.executable, 'tools/colab_backend.py', '--port', str(PORT)],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
_lines = queue.Queue()
threading.Thread(target=_drain, args=(backend.stdout, _lines), daemon=True).start()

TOKEN, ADAPTER = None, None
_deadline = time.time() + 300
while time.time() < _deadline and TOKEN is None:
    try:
        said = _lines.get(timeout=5)
    except queue.Empty:
        continue
    if said.startswith('BACKEND '):
        answer = json.loads(said[len('BACKEND '):])
        TOKEN, ADAPTER = answer['token'], answer['gpu']
    elif 'rror' in said:
        print(said)

if TOKEN is None:
    raise SystemExit('the fold service did not start; re-run cell 1')

# 🔴 THE ONE LINE WORTH READING BEFORE ANY FOLD. 'nvidia' and an architecture
# is the card; 'swiftshader' or 'llvmpipe' is the CPU wearing its clothes, and
# every number after that is a measurement of a fallback.
print('GPU :', ADAPTER.get('vendor'), ADAPTER.get('architecture'))
print('f16 :', ADAPTER.get('shaderF16'), '| subgroup matrix:', ADAPTER.get('subgroupMatrix'))
if 'swiftshader' in json.dumps(ADAPTER).lower() or 'llvmpipe' in json.dumps(ADAPTER).lower():
    print('*** that is the CPU renderer, not the card - re-run cell 1 ***')
else:
    print('ready: run the next cell as often as you like')

In [ ]:
#@title 3 · Fold { display-mode: "form" }
sequence = 'GWSTELEKHREELKEFLKKEGITLGFTNAEKQEQAQKLGLGKKVSPELLIKAFAILKK'  #@param {type:"string"}
model = 'af3'  #@param ["af3", "openbind0", "boltz2", "protenix2", "ef2-fast-600m", "monomer"]
modification = ''  #@param {type:"string"}
sampler_steps = 25  #@param {type:"slider", min:3, max:200, step:1}
recycles = 3  #@param {type:"slider", min:0, max:10, step:1}

import json, time, urllib.request

# `SEP@3`, or `SEP@3,PTR@11` - the spelling the page's own popup writes.
mods = []
for piece in (modification or '').split(','):
    piece = piece.strip()
    if not piece:
        continue
    code, _, at = piece.partition('@')
    mods.append({'code': code.strip().upper(), 'position': int(at)})

request = {
    'entities': [{'type': 'protein', 'value': sequence.strip().upper(),
                  'copies': 1, 'modifications': mods}],
    'model': model, 'steps': sampler_steps, 'recycles': recycles, 'timeout': 1500,
}

# 🔴 LOOPBACK, WITH NO TUNNEL AND NOTHING TO PASTE. The service listens on this
# runtime's own 127.0.0.1 and the only thing that can reach it is this cell, in
# the same machine. notebooks/colab-backend.ipynb is the other case - the same
# service behind a tunnel, driven from your own LocalFold page - and that one
# needs a token, because its door faces the internet.
print('folding', len(sequence.strip()), 'residues on', model, '...')
call = urllib.request.Request(
    'http://127.0.0.1:%d/fold?t=%s' % (PORT, TOKEN),
    data=json.dumps(request).encode(), headers={'content-type': 'application/json'})
with urllib.request.urlopen(call, timeout=1800) as answer:
    result = json.loads(answer.read())

if result.get('error'):
    print('***', result['error'], '|', result.get('status') or '')
else:
    print(result['status'])
    print('%d atoms | %d frames | %.1f s on the runtime'
          % (result['atoms'], result['frames'], result['ms'] / 1000))
    # ...and drawn HERE, in your browser, from the file the runtime made. The
    # picture is the only part of this that uses your own machine - which is
    # what a browser is for, and why embedding the whole site in a cell would
    # fold on your laptop with the T4 idle.
    import py2Dmol
    with open('/content/fold.pdb', 'w') as handle:
        handle.write(result['pdb'])
    view = py2Dmol.view()
    view.add_pdb('/content/fold.pdb')      # a PATH: there is no add_pdb_text
    view.show()
    print('saved to /content/fold.pdb')

---

### Driving this from your own LocalFold page instead

`notebooks/colab-backend.ipynb` is the same service with a cloudflared tunnel
in front of it: it prints one line you paste into LocalFold, and folds go to
this runtime while the page stays the page. That door faces the internet, so
it carries a token; this notebook needs none, because nothing but this kernel
can reach 127.0.0.1 here.